In [31]:
import os
import re
import sys
import glob
import uuid
import math
import time
import hashlib
import logging

import orjson
from tqdm import tqdm

import fitz  # PyMuPDF

In [32]:

# doc = fitz.open('./data/a_textbook_of_neuroanatomy.pdf')
# pages = []
# for i in range(len(doc)):
#     page = doc[i]
#     text = page.get_text("blocks", sort=True)
#     pages.append(text)
# doc.close()



In [33]:
# for i, box in enumerate(pages [62]):
#     print(i, box[-3])

In [34]:


_WORD_RE = re.compile(r"\w+|[^\w\s]", re.UNICODE)

def setup_logging(log_dir):
    os.makedirs(log_dir, exist_ok=True)
    log_path = os.path.join(log_dir, f"ingestion_{int(time.time())}.log")
    logging.basicConfig(
        filename=log_path,
        level=logging.INFO,
        format="%(asctime)s | %(levelname)s | %(message)s",
    )
    console = logging.StreamHandler(sys.stdout)
    console.setLevel(logging.INFO)
    console.setFormatter(logging.Formatter("%(levelname)s | %(message)s"))
    logging.getLogger().addHandler(console)
    return log_path

def tokenize(s):
    return _WORD_RE.findall(s)

def count_tokens(s):
    return len(tokenize(s))

def normalize_spaces(s):
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r"\u00A0", " ", s)
    s = re.sub(r" *\n *", "\n", s)
    s = re.sub(r"\n{3,}", "\n\n", s)
    return s.strip()


def clean_pdf(path, threshold=10):
    doc = fitz.open(path)
    base = os.path.basename(path)

    page_texts = []
    book_start = False
    book_end = False
    for i in range(len(doc)):
        try:
            page = doc[i]
            blocks = page.get_text("blocks") 
            if not blocks:
                page_texts.append("")
                continue

            mid_x = (page.rect.x0 + page.rect.x1) / 2
            left_blocks, right_blocks = [], []
            for b in blocks:
                if len(b) < 5:
                    continue
                x0, y0, x1, y1, txt = b[0], b[1], b[2], b[3], b[4] or ""
                if not txt.strip():
                    continue
                cx = (x0 + x1) / 2
                (left_blocks if cx < mid_x else right_blocks).append((y0, x0, txt))

            left_blocks.sort(key=lambda t: (t[0], t[1]))
            right_blocks.sort(key=lambda t: (t[0], t[1]))
            ordered = left_blocks + right_blocks

            kept_lines = []
            for j, (_, _, txt) in enumerate(ordered):
                plain = normalize_spaces(txt).lower()
                n_words = len(plain.split())
                print(plain)
                if not book_start and ("c h a p t e r" in plain or "gross anatomy of the brain" in plain):
                    book_start = True
                if plain in ["index", "i n d e x"]:
                    if j == 0 or len(doc) - i <= 20:
                        book_end = True
                        break
                if not book_start:
                    continue
                if n_words < threshold:
                    continue
             
                kept_lines.append(plain)

            page_texts.append("\n".join(kept_lines))
        except Exception as e:
            logging.exception(f"Failed to process blocks on page {i+1} of {base}: {e}")
            page_texts.append("")
        if book_end:
            break
    doc.close()
    return page_texts




def clean_and_parse_pdf(path):
    pages = clean_pdf(path)
    joined = "\n\n".join(pages)
    return joined


def split_into_paragraphs(text, threshold=10):
    paras = re.split(r"\n\s*\n", text)
    paras = [normalize_spaces(p) for p in paras if p and count_tokens(p) >= threshold]
    return paras

def recursive_split_by_tokens(text, max_tokens, overlap_tokens):
    toks = tokenize(text)
    if len(toks) <= max_tokens:
        return [text]
    chunks = []
    start = 0
    step = max_tokens - overlap_tokens
    while start < len(toks):
        end = min(len(toks), start + max_tokens)
        window = " ".join(toks[start:end])
        chunks.append(normalize_spaces(window))
        if end == len(toks):
            break
        start += step
    return chunks

def content_aware_chunk(text, max_tokens, overlap_tokens):
    paras = split_into_paragraphs(text)
    chunks = []
    current = []
    cur_tokens = 0

    for p in paras:
        ptoks = count_tokens(p)
        if ptoks > max_tokens:
            if current:
                chunks.append("\n\n".join(current))
                current = []
                cur_tokens = 0
            sub = recursive_split_by_tokens(p, max_tokens, overlap_tokens)
            chunks.extend(sub)
            continue

        if cur_tokens + ptoks + 1 <= max_tokens:
            current.append(p)
            cur_tokens += ptoks + 1
        else:
            if current:
                chunks.append("\n\n".join(current))
                tail = []
                tail_tokens = 0
                for para in reversed(current):
                    t = count_tokens(para)
                    tail.append(para)
                    tail_tokens += t
                    if tail_tokens >= overlap_tokens:
                        break
                current = list(reversed(tail))
                cur_tokens = sum(count_tokens(x) for x in current)
            current.append(p)
            cur_tokens += ptoks + 1

    if current:
        chunks.append("\n\n".join(current))

    final_chunks = []
    for ch in chunks:
        if count_tokens(ch) > max_tokens:
            final_chunks.extend(recursive_split_by_tokens(ch, max_tokens, overlap_tokens))
        else:
            final_chunks.append(ch)
    return final_chunks

def stable_hash(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

def deduplicate(chunks):
    seen = set()
    out = []
    for c in chunks:
        h = stable_hash(c)
        if h not in seen:
            out.append(c)
            seen.add(h)
    return out

def write_jsonl(path, rows):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "wb") as f:
        for r in rows:
            f.write(orjson.dumps(r, option=orjson.OPT_APPEND_NEWLINE))

def process_single_pdf(filepath, granularities, overlap_tokens, artifacts_dir):
    fname = os.path.basename(filepath)
    doc_id = f"doc_{uuid.uuid4().hex[:8]}"
    logging.info(f"Parsing: {fname}")

    cleaned = clean_and_parse_pdf(filepath)
    stats = {"clean_chars": len(cleaned)}
    logging.info(f"Cleaned {fname}: {stats}")

    cleaned_out = os.path.join(artifacts_dir, "clean", f"{os.path.splitext(fname)[0]}__clean.txt")
    os.makedirs(os.path.dirname(cleaned_out), exist_ok=True)
    with open(cleaned_out, "w", encoding="utf-8") as f:
        f.write(cleaned)

    for g in granularities:
        logging.info(f"Chunking {fname} at granularity {g} with overlap {overlap_tokens}")
        chunks = content_aware_chunk(cleaned, max_tokens=g, overlap_tokens=overlap_tokens)
        chunks = [normalize_spaces(c) for c in chunks]
        chunks = [c for c in chunks if count_tokens(c) >= 10]
        chunks = deduplicate(chunks)

        rows = []
        parent_hash = stable_hash(cleaned)
        for i, ch in enumerate(chunks):
            row = {
                "chunk_id": f"{doc_id}_{g}_{i:05d}",
                "doc_id": doc_id,
                "parent_doc_hash": parent_hash,
                "source_path": os.path.abspath(filepath),
                "source_basename": fname,
                "granularity_tokens": g,
                "overlap_tokens": overlap_tokens,
                "position": i,
                "num_positions": len(chunks),
                "text": ch,
                "n_tokens": count_tokens(ch),
                "prev_chunk_id": f"{doc_id}_{g}_{i-1:05d}" if i > 0 else None,
                "next_chunk_id": f"{doc_id}_{g}_{i+1:05d}" if i < len(chunks)-1 else None,
                "created_at": int(time.time()),
            }
            rows.append(row)

        out_path = os.path.join(artifacts_dir, "chunks", f"{g}_tokens_{fname[:-4]}.jsonl")
        write_jsonl(out_path, rows)
        logging.info(f"Wrote {len(rows)} chunks to {out_path}")

    return stats

def run_batch(input_dir, artifacts_dir, granularities, overlap_tokens):
    os.makedirs(artifacts_dir, exist_ok=True)
    log_path = setup_logging(os.path.join(artifacts_dir, "logs"))
    logging.info("=== Nervous System SME | Preprocessing & Chunking ===")
    logging.info(f"Input dir: {input_dir}")
    logging.info(f"Artifacts: {artifacts_dir}")
    logging.info(f"Granularities: {granularities}, overlap: {overlap_tokens}")
    pdfs = sorted(glob.glob(os.path.join(input_dir, "*.pdf")))
    if not pdfs:
        logging.warning("No PDFs found. Exiting.")
        return

    overall = {"files": 0, "clean_chars": 0, "errors": 0}
    for fp in tqdm(pdfs, desc="Processing PDFs", ncols=100):
        try:
            stats = process_single_pdf(fp, granularities, overlap_tokens, artifacts_dir)
            overall["files"] += 1
            overall["clean_chars"] += stats["clean_chars"]
        except Exception as e:
            overall["errors"] += 1
            logging.exception(f"Failed to process {fp}: {e}")

    logging.info(f"=== DONE | {overall} | Logs: {log_path}")



In [35]:
run_batch(input_dir="./data", artifacts_dir="./artifacts", granularities=[2048, 512, 128], overlap_tokens=64)

INFO | === Nervous System SME | Preprocessing & Chunking ===
INFO | === Nervous System SME | Preprocessing & Chunking ===
INFO | === Nervous System SME | Preprocessing & Chunking ===
INFO | === Nervous System SME | Preprocessing & Chunking ===
INFO | === Nervous System SME | Preprocessing & Chunking ===
INFO | === Nervous System SME | Preprocessing & Chunking ===
INFO | === Nervous System SME | Preprocessing & Chunking ===
INFO | Input dir: ./data
INFO | Input dir: ./data
INFO | Input dir: ./data
INFO | Input dir: ./data
INFO | Input dir: ./data
INFO | Input dir: ./data
INFO | Input dir: ./data
INFO | Artifacts: ./artifacts
INFO | Artifacts: ./artifacts
INFO | Artifacts: ./artifacts
INFO | Artifacts: ./artifacts
INFO | Artifacts: ./artifacts
INFO | Artifacts: ./artifacts
INFO | Artifacts: ./artifacts
INFO | Granularities: [2048, 512, 128], overlap: 64
INFO | Granularities: [2048, 512, 128], overlap: 64
INFO | Granularities: [2048, 512, 128], overlap: 64
INFO | Granularities: [2048, 512

Processing PDFs:   0%|                                                        | 0/3 [00:00<?, ?it/s]

INFO | Parsing: a_textbook_of_neuroanatomy.pdf
INFO | Parsing: a_textbook_of_neuroanatomy.pdf
INFO | Parsing: a_textbook_of_neuroanatomy.pdf
INFO | Parsing: a_textbook_of_neuroanatomy.pdf
INFO | Parsing: a_textbook_of_neuroanatomy.pdf
INFO | Parsing: a_textbook_of_neuroanatomy.pdf
INFO | Parsing: a_textbook_of_neuroanatomy.pdf
www.myuptodate.com
@mehrsyssupport
@mehrsyssupport
a textbook of neuroanatomy
dedication
to my father, antonios,
my mother, garifalia, and
my sister oursikía
for their contribution to my education
map
to my wife, roseann,
my daughter, jen, and
my mother, mary
lpg
xxv
the brain within its groove
runs evenly and true;
but let a splinter swerve,
‘t were easier for you
to put the water back
when ﬂoods have slit the hills,
and scooped a turnpike for themselves,
and blotted out the mills!
emily dickinson
a textbook of neuroanatomy
maria a. patestas
associate professor of anatomy
department of anatomy
des moines university
des moines, iowa
leslie p. gartner
professor of

Processing PDFs:  33%|████████████████                                | 1/3 [00:04<00:09,  4.93s/it]

INFO | Parsing: barr_human_nervous_system.pdf
INFO | Parsing: barr_human_nervous_system.pdf
INFO | Parsing: barr_human_nervous_system.pdf
INFO | Parsing: barr_human_nervous_system.pdf
INFO | Parsing: barr_human_nervous_system.pdf
INFO | Parsing: barr_human_nervous_system.pdf
INFO | Parsing: barr_human_nervous_system.pdf
barr’s
te n t h e d i t i o n
the human
nervous system
an anatomical viewpoint
barr’s
te n t h e d i t i o n
john a. kiernan, mb, chb, phd, dsc
professor emeritus
department of anatomy and cell biology
the university of western ontario
london, canada
and
nagalingam rajakumar, mb, bs, phd
associate professor
departments of psychiatry and anatomy and cell biology
the university of western ontario
london, canada
the human
nervous system
an anatomical viewpoint
acquisitions editor: crystal taylor
product manager: jenn verbiar
marketing manager: joy fisher-williams
vendor manager: bridgett dougherty
manufacturing coordinator: margie orzech
creative director: doug smock
compo

Processing PDFs:  67%|████████████████████████████████                | 2/3 [00:07<00:03,  3.63s/it]

INFO | Parsing: the_human_nervous_system_Charles_R_Noback.pdf
INFO | Parsing: the_human_nervous_system_Charles_R_Noback.pdf
INFO | Parsing: the_human_nervous_system_Charles_R_Noback.pdf
INFO | Parsing: the_human_nervous_system_Charles_R_Noback.pdf
INFO | Parsing: the_human_nervous_system_Charles_R_Noback.pdf
INFO | Parsing: the_human_nervous_system_Charles_R_Noback.pdf
INFO | Parsing: the_human_nervous_system_Charles_R_Noback.pdf
the human nervous system
structure and function
sixth edition
the human nervous system
structure and function
sixth edition
charles r. noback, phd
professor emeritus
department of anatomy and cell biology
college of physicians and surgeons
columbia university, new york, ny
norman l. strominger, phd
professor
center for neuropharmacology and neuroscience
department of surgery (otolaryngology)
the albany medical college
adjunct professor, division of biomedical science
university at albany institute for health and the environment
albany, ny
robert j. demarest
di

Processing PDFs: 100%|████████████████████████████████████████████████| 3/3 [00:10<00:00,  3.40s/it]

INFO | === DONE | {'files': 3, 'clean_chars': 3740013, 'errors': 0} | Logs: ./artifacts\logs\ingestion_1760966358.log
INFO | === DONE | {'files': 3, 'clean_chars': 3740013, 'errors': 0} | Logs: ./artifacts\logs\ingestion_1760966358.log
INFO | === DONE | {'files': 3, 'clean_chars': 3740013, 'errors': 0} | Logs: ./artifacts\logs\ingestion_1760966358.log
INFO | === DONE | {'files': 3, 'clean_chars': 3740013, 'errors': 0} | Logs: ./artifacts\logs\ingestion_1760966358.log
INFO | === DONE | {'files': 3, 'clean_chars': 3740013, 'errors': 0} | Logs: ./artifacts\logs\ingestion_1760966358.log
INFO | === DONE | {'files': 3, 'clean_chars': 3740013, 'errors': 0} | Logs: ./artifacts\logs\ingestion_1760966358.log
INFO | === DONE | {'files': 3, 'clean_chars': 3740013, 'errors': 0} | Logs: ./artifacts\logs\ingestion_1760966358.log
